# NYC Construction Permit Approval Duration
## Data Acquisition and Overview

This notebook loads permit issuance data from the NYC Department of Buildings 
via the NYC Open Data API and performs an initial overview of the dataset, 
including structure, variable types, and missingness patterns.

### Data Source

The NYC DOB Permit Issuance dataset tracks the lifecycle of every construction 
permit filed in New York City, from application through issuance. It is updated 
daily and is publicly available through NYC Open Data.

* **Source:** NYC Department of Buildings — Permit Issuance
* **Access:** NYC Open Data Socrata API
* **Dataset ID:** `ipu4-2q9a`

#### Libraries

The following libraries are used in this analysis:

* `pandas` — data manipulation and tabular analysis
* `sodapy` — Python client for the NYC Open Data Socrata API

In [2]:
import pandas as pd
from sodapy import Socrata

#### Data Acquisition

Data is pulled directly from the NYC Open Data API using the Socrata client. 
No manual download is required. The dataset ID `ipu4-2q9a` corresponds to the 
DOB Permit Issuance dataset, which tracks every construction permit filed in 
New York City from application through issuance.

A limit of 100,000 records is used to keep the pull manageable while retaining 
a large, representative sample for modeling.

In [13]:
import warnings
warnings.filterwarnings('ignore')

client = Socrata("data.cityofnewyork.us", None)

results = client.get("ipu4-2q9a", limit=100000)

df = pd.DataFrame.from_records(results)

#### Initial Data Overview

Basic inspection of the dataset; its shape, column names, and data types, to understand what variables are available before any cleaning or transformation.

In [6]:
print(df.shape)
print(df.dtypes)

(100000, 59)
borough                             object
bin__                               object
house__                             object
street_name                         object
job__                               object
job_doc___                          object
job_type                            object
self_cert                           object
block                               object
lot                                 object
community_board                     object
zip_code                            object
bldg_type                           object
work_type                           object
permit_status                       object
filing_status                       object
permit_type                         object
permit_sequence__                   object
permit_subtype                      object
filing_date                         object
issuance_date                       object
expiration_date                     object
job_start_date                      objec

#### Sample Records

A preview of the first few records to inspect the structure and values of key variables.

In [7]:
df[['filing_date', 'issuance_date', 'job_type', 'work_type', 
    'borough', 'bldg_type', 'permit_type', 'residential']].head(10)

,filing_date,issuance_date,job_type,work_type,borough,bldg_type,permit_type,residential
0,05/10/2022,05/10/2022,A3,EQ,MANHATTAN,2,EQ,NaN
1,05/10/2022,05/10/2022,A2,OT,STATEN ISLAND,1,EW,YES
2,05/10/2022,05/10/2022,A2,OT,STATEN ISLAND,1,EW,YES
3,05/10/2022,05/10/2022,A2,OT,STATEN ISLAND,1,EW,YES
4,05/10/2022,05/10/2022,A2,OT,STATEN ISLAND,1,EW,YES
5,05/10/2022,05/10/2022,NB,PL,BROOKLYN,2,PL,YES
6,05/10/2022,05/10/2022,A1,NaN,BROOKLYN,1,AL,YES
7,05/10/2022,05/10/2022,A2,OT,STATEN ISLAND,1,EW,YES
8,05/10/2022,05/10/2022,A3,EQ,BROOKLYN,2,EQ,YES
9,05/10/2022,05/10/2022,A1,NaN,BRONX,1,AL,YES


#### Missingness Assessment

Before modeling, assess missing data in the key variables to understand data quality and inform cleaning decisions.

In [8]:
cols_of_interest = ['filing_date', 'issuance_date', 'job_type', 'work_type', 
                    'borough', 'bldg_type', 'permit_type', 'residential']

missing = df[cols_of_interest].isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})

,missing_count,missing_pct
filing_date,0,0.00
issuance_date,1574,1.57
job_type,0,0.00
work_type,28672,28.67
borough,0,0.00
bldg_type,596,0.60
permit_type,0,0.00
residential,45644,45.64


#### Missingness Discussion

Key observations:

* `issuance_date` is missing in 1.57% of records, corresponding to permits that were filed but never issued. These records will be excluded from modeling since approval duration cannot be computed.
* `work_type` is missing in 28.67% of records. Rather than dropping these rows, missing values will be retained as a distinct category, as the absence of a work type may itself be informative.
* `residential` is missing in 45.64% of records. This likely reflects structural missingness — certain job or permit types may not require this field. This will be explored further during feature engineering.
* `bldg_type` is missing in 0.60% of records and will be dropped.

In [9]:
df.groupby('job_type')['residential'].apply(lambda x: x.isnull().mean().round(3))

job_type
A1    0.332
A2    0.481
A3    0.688
DM    1.000
NB    0.299
SG    1.000
Name: residential, dtype: float64

Missingness in `residential` is structural rather than random. Demolition (`DM`) 
and Sign (`SG`) permits are missing this field in 100% of cases, while other job 
types show varying rates. This pattern suggests the field is not applicable to 
certain permit types rather than being a data quality issue. Missing values will 
be retained as a distinct category during feature engineering.

#### Summary

This notebook established a direct connection to the NYC Open Data API and loaded 
100,000 DOB permit records for analysis. Initial inspection identified 59 variables, 
of which a focused subset will be used for modeling: `filing_date`, `issuance_date`, 
`job_type`, `work_type`, `borough`, `bldg_type`, `permit_type`, and `residential`.

Missingness assessment revealed that `issuance_date` is missing in 1.57% of records 
and `residential` is structurally missing for certain job types. These will be 
addressed in the following notebook.

Data cleaning, feature engineering, and construction of the response variable 
(`approval_duration`) are covered in Notebook 2.